# Shrub Detection Pipeline

End-to-end notebook for the V12 XGBoost shrub detection pipeline.

**Stages:**
1. Logging setup
2. Environment & package check
3. Artifact & input file check
4. Label Studio export (patch export for manual review)
5. SAM annotation summary
6. V12 XGBoost training
7. V12 prediction â€” all 6 sites
8. Results visualisation
9. View run log

> **Run environment:** WSL Ubuntu with `venv_linux` activated, or any Linux Python 3.10+ env.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from pathlib import Path

BASE = Path(".")

#  step definitions 
STEPS = [
    {
        "num":    "1",
        "script": "process_shrub_masks.py",
        "title":  "TLS - NAIP Mask Rasters",
        "desc":   "Aligns TLS point clouds to NAIP grid\nand rasterizes shrub footprints",
        "inputs": "TLS .laz  +  NAIP rasters",
        "output": "shrub mask rasters",
        "color":  "#4e79a7",
        "check":  None,
    },
    {
        "num":    "2",
        "script": "Patch Extraction",
        "title":  "128--128 px Patch Tiling",
        "desc":   "Tiles NAIP+CHM rasters into\n5-channel .npy patches",
        "inputs": "NAIP + CHM rasters",
        "output": "shrub_train.json\nshrub_val.json + *.npy",
        "color":  "#59a14f",
        "check":  BASE / "detectron2_dataset/shrub_train.json",
    },
    {
        "num":    "3",
        "script": "export_patches_for_labeling.py",
        "title":  "Label Studio Export",
        "desc":   "Exports patches as PNGs with\nexisting annotations for manual review",
        "inputs": "shrub_train.json / shrub_val.json",
        "output": "label_studio_images/\nlabel_studio_import.json",
        "color":  "#f28e2b",
        "check":  BASE / "label_studio_images/label_studio_import.json",
    },
    {
        "num":    "4",
        "script": "sam_annotate.py",
        "title":  "SAM Annotation Refinement",
        "desc":   "Pass 1: refines circular masks with SAM\nPass 2: AMG hard-negative mining",
        "inputs": "shrub_train.json + SAM checkpoint",
        "output": "shrub_train_sam.json\nshrub_val_sam.json",
        "color":  "#e15759",
        "check":  BASE / "detectron2_dataset/shrub_train_sam.json",
    },
    {
        "num":    "5",
        "script": "train_shrub_v12.py",
        "title":  "XGBoost Training (V12)",
        "desc":   "Pass 1: feature importance ranking\nPass 2: Optuna NSGA-II tuning",
        "inputs": "shrub_train_sam.json + *.npy",
        "output": "shrub_classifier_v12.joblib\nv12_model_meta.json",
        "color":  "#b07aa1",
        "check":  BASE / "shrub_classifier_v12.joblib",
    },
    {
        "num":    "6",
        "script": "predict_v12_all_sites.py",
        "title":  "V12 Inference - All Sites",
        "desc":   "Computes 20 features per pixel,\napplies XGBoost at threshold 0.544",
        "inputs": "NAIP + CHM rasters (6 sites)",
        "output": "*_v12_prob.tif\n*_v12_mask.tif",
        "color":  "#76b7b2",
        "check":  BASE / "predictions_v12",
    },
]

#  layout 
EXTRA_H  = 3.5                    # extra space below step 6 for fork + AI Agent
fig_h    = len(STEPS) * 1.55 + 0.6 + EXTRA_H
fig, ax  = plt.subplots(figsize=(13, fig_h))
ax.set_xlim(0, 13)
ax.set_ylim(0, fig_h)
ax.axis("off")

BOX_X, BOX_W, BOX_H = 0.35, 8.5, 1.15
STEP_H   = 1.55
Y_START  = fig_h - 0.55

ax.set_title("Shrubwise Detection Pipeline - Step Overview",
             fontsize=13, fontweight="bold", pad=10)

for i, s in enumerate(STEPS):
    y_top  = Y_START - i * STEP_H
    y_cent = y_top - BOX_H / 2

    #  status dot 
    if s["check"] is not None:
        done = (Path(s["check"]).exists() if isinstance(s["check"], (str, Path))
                else s["check"])
        dot_color = "#2ca02c" if done else "#d62728"
        dot_label = "- done" if done else "--- missing"
    else:
        dot_color, dot_label = "#aec7e8", ""

    #  main box 
    box = FancyBboxPatch((BOX_X, y_top - BOX_H), BOX_W, BOX_H,
                         boxstyle="round,pad=0.04",
                         facecolor=s["color"], edgecolor="white",
                         linewidth=1.5, alpha=0.92, zorder=2)
    ax.add_patch(box)

    # step number circle
    circ = plt.Circle((BOX_X + 0.42, y_cent), 0.28,
                       color="white", zorder=3)
    ax.add_patch(circ)
    ax.text(BOX_X + 0.42, y_cent, s["num"],
            ha="center", va="center", fontsize=11,
            fontweight="bold", color=s["color"], zorder=4)

    # script name
    ax.text(BOX_X + 0.88, y_cent + 0.22, s["script"],
            ha="left", va="center", fontsize=9.5,
            fontweight="bold", color="white", zorder=3)

    # description
    ax.text(BOX_X + 0.88, y_cent - 0.16, s["desc"],
            ha="left", va="center", fontsize=7.8,
            color="white", alpha=0.93, zorder=3, linespacing=1.4)

    #  I/O sidebar 
    io_x = BOX_X + BOX_W + 0.22
    ax.text(io_x, y_cent + 0.26, f"IN:  {s['inputs']}",
            ha="left", va="center", fontsize=7.2,
            color="#555555", style="italic")
    ax.text(io_x, y_cent - 0.18, f"OUT: {s['output']}",
            ha="left", va="center", fontsize=7.2,
            color="#222222", fontweight="bold", linespacing=1.3)

    # status dot + label
    if dot_label:
        ax.plot(BOX_X + BOX_W - 0.22, y_cent, "o",
                color=dot_color, markersize=8, zorder=4)
        ax.text(BOX_X + BOX_W - 0.22, y_cent - 0.32, dot_label,
                ha="center", va="center", fontsize=6.5,
                color=dot_color, fontweight="bold", zorder=4)

    #  connecting arrow (between main steps) 
    if i < len(STEPS) - 1:
        arr_x  = BOX_X + BOX_W / 2
        arr_y0 = y_top - BOX_H - 0.01
        arr_y1 = arr_y0 - (STEP_H - BOX_H) + 0.01
        ax.annotate("", xy=(arr_x, arr_y1), xytext=(arr_x, arr_y0),
                    arrowprops=dict(arrowstyle="-|>", color="#888888",
                                   lw=1.8, mutation_scale=16))

#  fork section below step 6 
step6_y_top = Y_START - 5 * STEP_H
step6_bot   = step6_y_top - BOX_H
step6_cx    = BOX_X + BOX_W / 2    # center x of main boxes - 4.6

# Stem arrow down from step 6
stem_y1 = step6_bot - 0.55
ax.annotate("", xy=(step6_cx, stem_y1), xytext=(step6_cx, step6_bot),
            arrowprops=dict(arrowstyle="-", color="#888888", lw=1.8))

# Horizontal fork bar
fork_y = stem_y1
ax.plot([1.8, step6_cx * 2 - 1.8], [fork_y, fork_y], color="#888888", lw=1.8)

# Fork labels
ax.text(1.8, fork_y + 0.12, "Known sites (batch)",
        ha="center", va="bottom", fontsize=8, color="#555555", style="italic")
ax.text(step6_cx * 2 - 1.8, fork_y + 0.12, "New AOI (any location)",
        ha="center", va="bottom", fontsize=8, color="#a07000", style="italic", fontweight="bold")

#  left branch: batch output note 
batch_x = 1.8
ax.annotate("", xy=(batch_x, fork_y - 0.65), xytext=(batch_x, fork_y),
            arrowprops=dict(arrowstyle="-|>", color="#76b7b2", lw=1.5, mutation_scale=14))

batch_box = FancyBboxPatch((0.2, fork_y - 1.7), 3.2, 0.95,
                           boxstyle="round,pad=0.06",
                           facecolor="#e8f5f5", edgecolor="#76b7b2",
                           linewidth=1.5, alpha=0.95, zorder=2)
ax.add_patch(batch_box)
ax.text(1.8, fork_y - 1.225, "predictions_v12/<site>/",
        ha="center", va="center", fontsize=8, fontweight="bold", color="#2a7a7a")
ax.text(1.8, fork_y - 1.55, "*_v12_prob.tif    *_v12_mask.tif",
        ha="center", va="center", fontsize=7.2, color="#444444")

#  right branch: AI Agent 
AGENT_CX = step6_cx * 2 - 1.8     # mirror of batch_x around step6_cx
ax.annotate("", xy=(AGENT_CX, fork_y - 0.65), xytext=(AGENT_CX, fork_y),
            arrowprops=dict(arrowstyle="-|>", color="#d4aa00", lw=2.0,
                            mutation_scale=14))

AGENT_X, AGENT_W, AGENT_H = 4.9, 7.7, 1.7
agent_box = FancyBboxPatch((AGENT_X, fork_y - 2.55), AGENT_W, AGENT_H,
                           boxstyle="round,pad=0.07",
                           facecolor="#fff8e1", edgecolor="#d4aa00",
                           linewidth=2.2, alpha=0.97, zorder=2)
ax.add_patch(agent_box)

# Step 7 number circle
agent_cy = fork_y - 2.55 + AGENT_H / 2
circ7 = plt.Circle((AGENT_X + 0.47, agent_cy), 0.30,
                   color="#d4aa00", zorder=3)
ax.add_patch(circ7)
ax.text(AGENT_X + 0.47, agent_cy, "7",
        ha="center", va="center", fontsize=11,
        fontweight="bold", color="white", zorder=4)

ax.text(AGENT_X + 0.95, agent_cy + 0.38,
        "autonomous_shrub_agent.py",
        ha="left", va="center", fontsize=9.5, fontweight="bold",
        color="#7a5c00", zorder=3)
ax.text(AGENT_X + 0.95, agent_cy + 0.08,
        "AI AGENT - Predict New Sites",
        ha="left", va="center", fontsize=8.8,
        color="#a07000", zorder=3, style="italic")
ax.text(AGENT_X + 0.95, agent_cy - 0.30,
        "IN:   GeoJSON AOI file\nAUTO: downloads NAIP (Planetary Computer) + CHM (GEE)\nOUT:  naip.tif / chm.tif / predicted_shrub_v12.tif",
        ha="left", va="center", fontsize=7.4,
        color="#5a4000", zorder=3, linespacing=1.45)

# Status dot for AI agent
agent_check = BASE / "Model_Prediction/naip.tif"
done7 = agent_check.exists()
dot7_color = "#2ca02c" if done7 else "#d62728"
dot7_label = "- done" if done7 else "--- missing"
ax.plot(AGENT_X + AGENT_W - 0.25, agent_cy, "o",
        color=dot7_color, markersize=8, zorder=4)
ax.text(AGENT_X + AGENT_W - 0.25, agent_cy - 0.32, dot7_label,
        ha="center", va="center", fontsize=6.5,
        color=dot7_color, fontweight="bold", zorder=4)

#  legend 
legend_elements = [
    mpatches.Patch(color="#2ca02c", label="Output exists"),
    mpatches.Patch(color="#d62728", label="Output missing"),
    mpatches.Patch(color="#aec7e8", label="No check"),
    mpatches.Patch(facecolor="#fff8e1", edgecolor="#d4aa00", linewidth=1.5,
                   label="AI Agent (Step 7)"),
]
ax.legend(handles=legend_elements, loc="lower left",
          fontsize=8, framealpha=0.85, edgecolor="#cccccc")

plt.tight_layout()
diagram_path = BASE / "predictions_v12" / "pipeline_diagram.png"
diagram_path.parent.mkdir(exist_ok=True)
plt.savefig(diagram_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {diagram_path}")


---
## 1. Logging Setup
Creates a timestamped log file in `logs/` that mirrors every stage of this run.

In [ ]:
import logging
import sys
import time
import json
import os
from pathlib import Path
from datetime import datetime

BASE    = Path(".")
LOG_DIR = BASE / "logs"
LOG_DIR.mkdir(exist_ok=True)

RUN_TS   = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = LOG_DIR / f"{RUN_TS}_pipeline.log"

logger = logging.getLogger("shrub_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s  %(levelname)-7s  %(message)s", "%H:%M:%S")
)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(logging.Formatter("%(asctime)s  %(message)s", "%H:%M:%S"))
logger.addHandler(stream_handler)

def log(msg="", level="info"):
    getattr(logger, level)(str(msg))

def log_section(title):
    log("=" * 60)
    log(f"  {title}")
    log("=" * 60)

log_section(f"Shrub Pipeline Run  -  {RUN_TS}")
log(f"Log file : {LOG_PATH.resolve()}")
log(f"Python   : {sys.version.split()[0]}")
print(f"\n Log file: {LOG_PATH.resolve()}")


---
## 2. Environment & Package Check

In [ ]:
import importlib
import subprocess, sys

# ------------------------------------------------------------------ options --
# Set True to trigger optional actions below
TRY_GEE_AUTH = False   # run ee.Authenticate() + ee.Initialize()
INSTALL_SAM  = False   # pip-install segment-anything if missing
# ----------------------------------------------------------------------------

REQUIRED = {
    "numpy":              "numpy",
    "pandas":             "pandas",
    "scipy":              "scipy",
    "sklearn":            "scikit-learn",
    "xgboost":            "xgboost",
    "optuna":             "optuna",
    "joblib":             "joblib",
    "rasterio":           "rasterio",
    "geopandas":          "geopandas",
    "shapely":            "shapely",
    "cv2":                "opencv-python",
    "matplotlib":         "matplotlib",
    "torch":              "torch",
    "torchvision":        "torchvision",
    "PIL":                "Pillow",
    "planetary_computer": "planetary-computer",
    "pystac_client":      "pystac-client",
}
OPTIONAL = {
    "ee":               "earthengine-api",
    "segment_anything": "segment-anything",
}

log_section("2. Environment Check")

# -- Required packages -------------------------------------------------------
all_ok = True
for mod, pkg in REQUIRED.items():
    try:
        m   = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        log(f"  OK    {pkg:<25} {ver}")
    except ImportError:
        log(f"  MISS  {pkg:<25} MISSING - pip install {pkg}", level="warning")
        all_ok = False

# -- Optional packages -------------------------------------------------------
log("")
sam_installed = False
for mod, pkg in OPTIONAL.items():
    try:
        m   = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        log(f"  OK    {pkg:<25} {ver}  [optional]")
        if mod == "segment_anything":
            sam_installed = True
    except ImportError:
        log(f"  -     {pkg:<25} not installed [optional]")

# -- Install segment-anything (optional) ------------------------------------
if INSTALL_SAM and not sam_installed:
    log("")
    log("  Installing segment-anything from Meta GitHub...")
    _res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/facebookresearch/segment-anything.git"],
        capture_output=True, text=True
    )
    if _res.returncode == 0:
        log("  segment-anything installed successfully.")
        log("  NOTE: download checkpoint manually:")
        log("    wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth")
    else:
        log(f"  Install failed:\n{_res.stderr[:400]}", level="error")
elif INSTALL_SAM and sam_installed:
    log("  segment-anything already installed -- skipping.")

# -- GPU / CUDA --------------------------------------------------------------
import torch
cuda_info = (
    f"CUDA {torch.version.cuda} - {torch.cuda.get_device_name(0)}"
    if torch.cuda.is_available() else "CPU only"
)
log(f"\n  GPU  : {cuda_info}")
status = "OK - all required packages present" if all_ok else "ERROR - missing packages"
log(f"  ENV  : {status}")

# -- Google Earth Engine authentication (optional) --------------------------
log("")
if TRY_GEE_AUTH:
    try:
        import ee
        log("  GEE  : Authenticating... (browser window may open)")
        ee.Authenticate()
        ee.Initialize(project="ee-sefakarabas")
        log("  GEE  : Authenticated and initialised OK (project=ee-sefakarabas)")
    except Exception as _e:
        log(f"  GEE  : Authentication failed -- {_e}", level="warning")
        log("         Run: earthengine authenticate  (in terminal)", level="warning")
else:
    try:
        import ee
        ee.Initialize(project="ee-sefakarabas")
        log("  GEE  : Already authenticated (initialised OK)")
    except Exception:
        log("  GEE  : Not authenticated (set TRY_GEE_AUTH=True to authenticate)")


---
## 3. Artifact & Input File Check

In [ ]:
ARTIFACTS = {
    "V12 classifier":        BASE / "shrub_classifier_v12.joblib",
    "V12 scaler":            BASE / "shrub_scaler_v12.joblib",
    "V12 meta":              BASE / "v12_model_meta.json",
    "SAM train JSON":        BASE / "detectron2_dataset/shrub_train_sam.json",
    "SAM val JSON":          BASE / "detectron2_dataset/shrub_val_sam.json",
    "SAM checkpoint (opt.)": BASE / "sam_vit_h_4b8939.pth",
}

SITES = {
    "Calaveras_Big_trees": {"naip": "calaveras_big_trees_1m_naip_2022.tif",   "chm": "calaveras_big_trees_canopy_height_1m.tif"},
    "DL_Bliss":            {"naip": "dl_bliss_1m_naip_2022.tif",              "chm": "dl_bliss_canopy_height_1m.tif"},
    "Independence_Lake":   {"naip": "independence_lake_1m_naip_2022.tif",     "chm": "independence_lake_canopy_height_1m.tif"},
    "Pacific_Union":       {"naip": "pacific_union_college_1m_naip_2022.tif", "chm": "pacific_union_canopy_height_1m.tif"},
    "Sedgwick":            {"naip": "sedgwick_1m_naip_2022.tif",              "chm": "sedgwick_canopy_height_1m.tif"},
    "Shaver_Lake":         {"naip": "shaver_lake_1m_naip_2022.tif",           "chm": "shaver_lake_canopy_height_1m.tif"},
}

log_section("3. Artifact & Input File Check")

log("Model artifacts:")
MODEL_READY = True
for name, path in ARTIFACTS.items():
    exists = path.exists()
    size   = f"{path.stat().st_size/1e6:.1f} MB" if exists else ""
    mark   = "OK  " if exists else ("OPT " if "opt" in name else "MISS")
    if not exists and "opt" not in name:
        MODEL_READY = False
    log(f"  {mark}  {name:<30} {size}")

log("")
log("Site input files:")
sites_ready = []
for site, cfg in SITES.items():
    naip_dir = BASE / site / "NAIP_3DEP_product"
    naip_ok  = (naip_dir / cfg["naip"]).exists()
    chm_ok   = (naip_dir / cfg["chm"]).exists()
    if naip_ok and chm_ok:
        sites_ready.append(site)
        log(f"  OK    {site:<25} NAIP -  CHM -")
    else:
        log(f"  MISS  {site:<25} NAIP {'-' if naip_ok else '---'}  CHM {'-' if chm_ok else '---'}",
            level="warning")

log(f"\n  {len(sites_ready)}/6 sites ready  |  model_ready={MODEL_READY}")


---
## Step 1 â€” process_shrub_masks.py
**TLS â†’ NAIP Mask Rasters**

Aligns Terrestrial LiDAR Scan (TLS) point clouds to the NAIP grid and rasterizes individual shrub footprints into per-polygon mask TIFs.

| Input | Output |
|---|---|
| TLS `.laz` files (aligned) | `mask_outputs_sprint4/<site>_mask.tif` (per polygon) |
| NAIP rasters | Shrub footprint rasters at 1 m/px |
| `shrub_lists_revised/` CSVs | â€” |

In [ ]:
log_section("Step 1. TLS - NAIP Mask Rasters (process_shrub_masks.py)")

#  Print log file if it exists 
MASK_LOG = BASE / "process_shrub_masks_log.txt"
if MASK_LOG.exists():
    log_text = MASK_LOG.read_text(encoding="utf-8", errors="replace")
    print("=" * 70)
    print("  process_shrub_masks_log.txt")
    print("=" * 70)
    print(log_text[-8000:] if len(log_text) > 8000 else log_text)
    print("=" * 70 + "\n")
else:
    log(f"  process_shrub_masks_log.txt not found - log will appear here after running the script",
        level="warning")

#  Per-site mask output summary 
log("Mask output status per site (mask_outputs/):")
log(f"  {'Site':<25} {'Sprint4 masks':>14} {'Shrub lists':>13} {'TLS .laz':>10}")
log("-" * 68)

total_masks = 0
for site in SITES:
    sprint4_dir   = BASE / site / "mask_outputs"
    shrub_list_dir = BASE / site / "shrub_lists_revised"
    if not shrub_list_dir.exists():
        shrub_list_dir = BASE / site / "shrub_lists"

    # Count mask TIFs (exclude .aux.xml)
    mask_tifs = list(sprint4_dir.glob("*_mask.tif")) if sprint4_dir.exists() else []
    n_masks   = len(mask_tifs)
    total_masks += n_masks

    # Count shrub list CSVs
    shrub_csvs = list(shrub_list_dir.glob("*.csv")) if shrub_list_dir.exists() else []
    n_lists    = len(shrub_csvs)

    # Count aligned TLS .laz files
    tls_dir  = BASE / site / "aligned_TLS"
    tls_lazs = list(tls_dir.glob("*.laz")) if tls_dir.exists() else []
    n_laz    = len(tls_lazs)

    mark = "OK  " if n_masks > 0 else "MISS"
    log(f"  {mark}  {site:<25} {n_masks:>8} masks   {n_lists:>6} CSVs   {n_laz:>5} .laz")

log(f"\n  Total mask TIFs across all sites: {total_masks}")
log(f"  Outputs live in: <Site>/mask_outputs_sprint4/")


---
## 4. SAM Annotation Summary

---
## 4. Label Studio Export
Exports patches as PNGs for manual review/correction in Label Studio â€” run **before** SAM annotation.

In [ ]:
log_section("4. Label Studio Export")

LS_DIR     = BASE / "label_studio_images"
LS_JSON    = LS_DIR / "label_studio_import.json"
EXPORT_LOG = BASE / "export_patches_log.txt"

# Show export log if it exists
if EXPORT_LOG.exists():
    log_text = EXPORT_LOG.read_text(encoding="utf-8", errors="replace")
    print("=" * 70)
    print(f"  export_patches_log.txt")
    print("=" * 70)
    print(log_text[-6000:] if len(log_text) > 6000 else log_text)
    print("=" * 70 + "\n")
else:
    log(f"  export_patches_log.txt not found - run export_patches_for_labeling.py to generate", level="warning")

# Show output summary
if LS_JSON.exists():
    import json as _json
    with open(LS_JSON) as f:
        tasks = _json.load(f)
    sites_in_export = sorted(set(t["data"]["site"] for t in tasks))
    log(f"  Label Studio import: {len(tasks)} tasks across {len(sites_in_export)} sites")
    log(f"  JSON path : {LS_JSON}")
    for site in sites_in_export:
        site_tasks = [t for t in tasks if t["data"]["site"] == site]
        annotated  = sum(1 for t in site_tasks if t["data"]["has_shrub_gt"])
        log(f"    {site:<25} {len(site_tasks):>4} patches  ({annotated} pre-annotated)")
else:
    log(f"  label_studio_import.json not found at {LS_JSON}", level="warning")
    log("  Run: python export_patches_for_labeling.py")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

SAM_VIZ_DIR = BASE / "scratch" / "sam_viz"

if not SAM_VIZ_DIR.exists():
    log(f"  sam_viz directory not found at {SAM_VIZ_DIR}", level="warning")
else:
    png_files = sorted(SAM_VIZ_DIR.glob("*.png"))
    log(f"  Found {len(png_files)} images in scratch/sam_viz/")

    #  sam_train_grid first (full width) 
    grid_path = SAM_VIZ_DIR / "sam_train_grid.png"
    if grid_path.exists():
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(mpimg.imread(grid_path))
        ax.axis("off")
        ax.set_title("SAM Annotations - Train Split Grid", fontsize=12, fontweight="bold", pad=8)
        plt.tight_layout()
        plt.show()

    #  individual tile images 
    tile_imgs = [p for p in png_files if p.name != "sam_train_grid.png"]

    if tile_imgs:
        n     = len(tile_imgs)
        ncols = 4
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.8 * nrows))
        axes = axes.flatten() if nrows > 1 else list(axes) if ncols > 1 else [axes]

        for i, img_path in enumerate(tile_imgs):
            img = mpimg.imread(img_path)
            axes[i].imshow(img)
            axes[i].axis("off")
            axes[i].set_title(img_path.stem, fontsize=7.5, pad=3)

        for j in range(len(tile_imgs), len(axes)):
            axes[j].set_visible(False)

        plt.suptitle("SAM Annotation Visualisations - Individual Tiles", fontsize=11, fontweight="bold", y=1.01)
        plt.tight_layout()
        plt.show()

    log(f"  Displayed {len(png_files)} SAM viz images from {SAM_VIZ_DIR}")


In [ ]:
import subprocess, sys
from pathlib import Path

log_section("Generate All SAM Tile PNGs")

SCRIPT  = BASE / "generate_all_sam_viz.py"
OUT_DIR = BASE / "sam_viz_all"

# -- Tile count table -------------------------------------------------
SITE_TILES = [
    ("Calaveras_Big_trees", 463),
    ("DL_Bliss",             40),
    ("Independence_Lake",   132),
    ("Pacific_Union",        53),
    ("Sedgwick",            151),
    ("Shaver_Lake",          40),
]
total_tiles = sum(n for _, n in SITE_TILES)

log("SAM tile counts per site (train + val combined):")
log(f"  +---------------------+-------+")
log(f"  | {'Site':<19} | {'Tiles':>5} |")
log(f"  +---------------------+-------+")
for i, (site, n) in enumerate(SITE_TILES):
    log(f"  | {site:<19} | {n:>5} |")
    if i < len(SITE_TILES) - 1:
        log(f"  +---------------------+-------+")
log(f"  +---------------------+-------+")
log(f"  | {'Total':<19} | {total_tiles:>5} |")
log(f"  +---------------------+-------+")
log("")

# -- Run script if outputs not yet present ----------------------------
if not SCRIPT.exists():
    log(f"  Script not found: {SCRIPT}", level="warning")
elif not OUT_DIR.exists():
    log("  Running generate_all_sam_viz.py (both splits, all sites)...")
    import time as _time
    t0 = _time.time()
    result = subprocess.run(
        [sys.executable, str(SCRIPT),
         "--out-dir", str(OUT_DIR),
         "--splits", "train", "val",
         "--dpi", "100", "--grid-cols", "6"],
        capture_output=True, text=True
    )
    elapsed = _time.time() - t0
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        log(f"  ERROR (exit {result.returncode})", level="error")
    else:
        log(f"  Done in {elapsed:.0f}s")
else:
    log(f"  sam_viz_all/ already exists ({OUT_DIR.resolve()}) -- skipping re-generation")

# -- Show summary grid -------------------------------------------------
summary_grid = OUT_DIR / "summary_grid.png"
if summary_grid.exists():
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(mpimg.imread(summary_grid))
    ax.axis("off")
    ax.set_title(
        f"SAM Viz -- summary_grid.png  (4 patches per site x {len(SITE_TILES)} sites)\n"
        f"Total tiles in dataset: {total_tiles}  |  green=shrub  red=tree  blue=rock",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
    log(f"  Outputs: {OUT_DIR.resolve()}")
else:
    log(f"  summary_grid.png not found at {OUT_DIR} -- run generate_all_sam_viz.py first",
        level="warning")


In [ ]:
log_section("4. SAM Annotation")

# Print sam_annotation_log.txt FIRST
SAM_LOG = BASE / "sam_annotation_log.txt"
if SAM_LOG.exists():
    log_text = SAM_LOG.read_text(encoding="utf-8", errors="replace")
    print("=" * 70)
    print(f"  sam_annotation_log.txt")
    print("=" * 70)
    print(log_text[-8000:] if len(log_text) > 8000 else log_text)
    print("=" * 70 + "\n")
else:
    log(f"  sam_annotation_log.txt not found at {SAM_LOG.resolve()}", level="warning")

# Then show annotation counts summary
sam_stats_path = BASE / "detectron2_dataset/sam_stats.json"
if sam_stats_path.exists():
    with open(sam_stats_path) as f:
        stats = json.load(f)
    for split, s in stats.items():
        log(f"  {split.upper()}:")
        for k, v in s.items():
            log(f"    {k:<20} {v}")

for name, path in [("Train", BASE / "detectron2_dataset/shrub_train_sam.json"),
                   ("Val",   BASE / "detectron2_dataset/shrub_val_sam.json")]:
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        all_anns = [a for d in data for a in d.get("annotations", [])]
        shrubs = sum(1 for a in all_anns if a.get("category_id") == 0)
        trees  = sum(1 for a in all_anns if a.get("category_id") == 1)
        rocks  = sum(1 for a in all_anns if a.get("category_id") == 2)
        log(f"  {name}: {len(data)} patches | shrub={shrubs} tree={trees} rock={rocks} total={len(all_anns)}")


---
## 5. V12 XGBoost Training
Skipped if model already exists. Set `FORCE_RETRAIN = True` to retrain from scratch.

In [ ]:
log_section("5. V12 Training")

# Print v12_train.log FIRST
TRAIN_LOG = BASE / "v12_train.log"
if TRAIN_LOG.exists():
    log_text = TRAIN_LOG.read_text(encoding="utf-8", errors="replace")
    print("=" * 70)
    print(f"  v12_train.log")
    print("=" * 70)
    print(log_text[-10000:] if len(log_text) > 10000 else log_text)
    print("=" * 70 + "\n")
else:
    log(f"  v12_train.log not found at {TRAIN_LOG.resolve()}", level="warning")

# Then show current model metrics
meta_path = BASE / "v12_model_meta.json"
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    log(f"  Model: {len(meta['features'])} features | threshold={meta['cls_threshold']}")
    for k, v in meta["metrics"].items():
        log(f"    {k:<12} {v:.4f}")
    log(f"  Top features: {meta['features'][:5]}")


---
## 5b. Feature Importance
Top-20 features selected by XGBoost Pass-1 importance from `v12_model_meta.json`, colour-coded by feature group.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

log_section("5b. Feature Importance")

META_PATH = BASE / "v12_model_meta.json"
with open(META_PATH) as _f:
    _meta = json.load(_f)

# Top-20 selected features (in training order) + their Pass-1 importance
_selected = set(_meta["features"])
_ranking  = [e for e in _meta["feature_selection"]["pass1_importance_ranking"]
             if e["feature"] in _selected]
_ranking  = sorted(_ranking, key=lambda e: e["importance"])  # ascending for barh

_features = [e["feature"] for e in _ranking]
_scores   = [e["importance"] for e in _ranking]

# Colour scheme by feature group
def _feat_color(name):
    if name.startswith("canopy_height"):
        return "#1565C0"   # dark blue  -- canopy height texture
    if name in ("canopy_shrub_clipped", "canopy_in_shrub_range"):
        return "#42A5F5"   # light blue -- canopy derived
    if name.startswith(("naip_red", "naip_green", "naip_nir", "naip_blue")):
        return "#2E7D32"   # dark green -- NAIP bands / texture
    return "#E65100"       # orange     -- spectral indices / texture

_colors = [_feat_color(f) for f in _features]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(_features, _scores, color=_colors, edgecolor="white", height=0.7)

# Value labels
for bar, score in zip(bars, _scores):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{score:.4f}", va="center", ha="left", fontsize=8)

# Legend
legend_items = [
    mpatches.Patch(color="#1565C0", label="Canopy height texture (mean/std)"),
    mpatches.Patch(color="#42A5F5", label="Canopy height derived"),
    mpatches.Patch(color="#2E7D32", label="NAIP band / texture"),
    mpatches.Patch(color="#E65100", label="Spectral index / texture"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9, framealpha=0.9)

ax.set_xlabel("XGBoost feature importance (Pass-1)", fontsize=10)
ax.set_title(
    f"V12 Top-{len(_features)} Feature Importance\n"
    f"(selected from {_meta['feature_selection']['candidate_count']} candidates)",
    fontsize=12, fontweight="bold"
)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, max(_scores) * 1.18)
plt.tight_layout()

feat_imp_path = OUT_ROOT / "v12_feature_importance.png"
plt.savefig(feat_imp_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
log(f"  Saved feature importance chart: {feat_imp_path}")

# Summary stats
_canopy_imp = sum(s for f, s in zip(_features, _scores)
                  if f.startswith("canopy_height") or
                  f in ("canopy_shrub_clipped", "canopy_in_shrub_range"))
log(f"  Canopy-height group cumulative importance: {_canopy_imp:.1%}")
log(f"  Top feature: {_features[-1]}  ({_scores[-1]:.4f})")


---
## 6. V12 Prediction â€” All Sites

In [ ]:
import joblib
import numpy as np
import rasterio

sys.path.insert(0, str(BASE))
from predict_raster_v12 import run_shrub_prediction_v12

model  = joblib.load(BASE / "shrub_classifier_v12.joblib")
scaler = joblib.load(BASE / "shrub_scaler_v12.joblib")
with open(BASE / "v12_model_meta.json") as f:
    meta = json.load(f)

OUT_ROOT = BASE / "predictions_v12"
OUT_ROOT.mkdir(exist_ok=True)

log_section("6. V12 Prediction")
log(f"  Model     : {type(model).__name__}")
log(f"  Features  : {len(meta['features'])}  threshold={meta['cls_threshold']}")
log(f"  Sites     : {sites_ready}")
log(f"  Output dir: {OUT_ROOT.resolve()}")
log("")

results = []
t_total = time.time()

for site in sites_ready:
    cfg      = SITES[site]
    naip_dir = BASE / site / "NAIP_3DEP_product"
    naip_p   = naip_dir / cfg["naip"]
    chm_p    = naip_dir / cfg["chm"]
    out_dir  = OUT_ROOT / site
    out_dir.mkdir(exist_ok=True)
    prefix   = site.lower().replace(" ", "_")
    mask_p   = out_dir / f"{prefix}_v12_mask.tif"

    log(f"  [{site}] starting ...")
    t0 = time.time()

    try:
        run_shrub_prediction_v12(
            naip_path=str(naip_p),
            chm_path=str(chm_p),
            output_path=str(mask_p),
        )
        elapsed = time.time() - t0

        with rasterio.open(mask_p) as src:
            binary = src.read(1)
            H, W   = src.height, src.width
        prob_p = out_dir / f"{prefix}_v12_prob.tif"
        with rasterio.open(prob_p) as src:
            prob = src.read(1)

        shrub_px = int((binary == 1).sum())
        r = {
            "site":     site,
            "size":     f"{H}x{W}",
            "shrub_ha": round(shrub_px / 1e4, 2),
            "p50":      round(float(np.percentile(prob, 50)), 3),
            "p95":      round(float(np.percentile(prob, 95)), 3),
            "elapsed":  round(elapsed, 1),
            "prob_map": prob,
            "mask":     binary,
            "status":   "OK",
        }
        results.append(r)
        log(f"  [{site}] OK - {H}x{W} px | "
            f"shrub={shrub_px:,} px ({r['shrub_ha']} ha) | "
            f"p50={r['p50']} p95={r['p95']} | {elapsed:.1f}s")

    except Exception as exc:
        elapsed = time.time() - t0
        log(f"  [{site}] ERROR after {elapsed:.1f}s: {exc}", level="error")
        results.append({"site": site, "status": "ERROR", "error": str(exc)})

total_elapsed = time.time() - t_total
log(f"\n  Total prediction time: {total_elapsed:.0f}s")

log("")
log(f"{'Site':<25} {'Size':>10} {'Shrub ha':>10} {'p50':>6} {'p95':>6} {'Time':>7} {'Status':>7}")
log("-" * 75)
total_ha = 0
for r in results:
    if r["status"] == "OK":
        total_ha += r["shrub_ha"]
        log(f"  {r['site']:<25} {r['size']:>10} {r['shrub_ha']:>10.2f} "
            f"{r['p50']:>6.3f} {r['p95']:>6.3f} {r['elapsed']:>6.1f}s {'OK':>7}")
    else:
        log(f"  {r['site']:<25} {'ERROR':>10}  {r.get('error','')}", level="error")
log("-" * 75)
ok_count = sum(1 for r in results if r["status"] == "OK")
log(f"  Total shrub area : {total_ha:.1f} ha across {ok_count}/{len(results)} sites")


---
## 7. Results Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

log_section("7. Visualisation -- Site by Site")

SHRUB_COLOR  = "#1db954"   # vivid green = shrub
NOSHRUB_COLOR = "#d0d0d0"  # light grey  = non-shrub
CMAP_BIN = ListedColormap([NOSHRUB_COLOR, SHRUB_COLOR])

def read_rgb(naip_path):
    with rasterio.open(naip_path) as src:
        r = src.read(1).astype(np.float32)
        g = src.read(2).astype(np.float32)
        b = src.read(3).astype(np.float32)
    rgb = np.stack([r, g, b], axis=-1)
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], [2, 98])
        rgb[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo), 0, 1) if hi > lo else 0
    return rgb

def draw_mask_panel(ax, mask, r_info, meta, fontsize=10):
    """Show binary mask as 2-colour image with legend and stats."""
    n_shrub = int(mask.sum())
    pct     = 100.0 * n_shrub / mask.size
    ax.imshow(mask, cmap=CMAP_BIN, vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    legend_elems = [
        mpatches.Patch(facecolor=SHRUB_COLOR,  edgecolor="none",
                       label=f"Shrub  {r_info['shrub_ha']} ha  ({pct:.1f}%)"),
        mpatches.Patch(facecolor=NOSHRUB_COLOR, edgecolor="#888",
                       label="Non-shrub"),
    ]
    ax.legend(handles=legend_elems, loc="lower left",
              fontsize=max(fontsize - 2, 6), framealpha=0.85,
              edgecolor="#aaa")
    ax.set_title(
        f"Binary Shrub Mask  (thr = {meta['cls_threshold']})\n"
        f"{r_info['shrub_ha']} ha shrub  |  {pct:.1f}% of raster",
        fontsize=fontsize
    )
    ax.axis("off")

ok_results = [r for r in results if r["status"] == "OK"]

# -- One figure per site --------------------------------------------------
site_paths = []
for r in ok_results:
    site   = r["site"]
    naip_p = BASE / site / "NAIP_3DEP_product" / SITES[site]["naip"]
    rgb    = read_rgb(naip_p)
    prob   = r["prob_map"]
    mask   = r["mask"]
    H, W   = mask.shape

    aspect  = H / W
    fig_w   = 15
    img_h   = fig_w / 3 * aspect
    fig_h   = max(img_h + 1.2, 3.5)

    fig, axes = plt.subplots(1, 3, figsize=(fig_w, fig_h))

    # Panel 1: NAIP RGB
    axes[0].imshow(rgb, aspect="auto")
    axes[0].set_title(f"{site}\nNAIP RGB", fontsize=10, fontweight="bold")
    axes[0].axis("off")

    # Panel 2: Probability map
    im = axes[1].imshow(prob, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_label("Shrub probability", fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    axes[1].set_title(f"V12 Probability\np50={r['p50']}  p95={r['p95']}", fontsize=10)
    axes[1].axis("off")

    # Panel 3: Binary mask (2-colour map + legend)
    draw_mask_panel(axes[2], mask, r, meta, fontsize=10)

    fig.suptitle(f"V12 XGBoost Shrub Detection -- {site}  ({H}x{W} px)",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()

    site_png = OUT_ROOT / site / f"{site.lower()}_v12_overview.png"
    plt.savefig(site_png, dpi=150, bbox_inches="tight", facecolor="white")
    site_paths.append(site_png)
    plt.show()
    log(f"  [{site}] saved overview: {site_png.name}")

# -- Combined overview -----------------------------------------------------
n = len(ok_results)
fig2, axes2 = plt.subplots(n, 3, figsize=(15, 4 * n))
if n == 1:
    axes2 = [axes2]
for row, r in enumerate(ok_results):
    site   = r["site"]
    naip_p = BASE / site / "NAIP_3DEP_product" / SITES[site]["naip"]
    rgb    = read_rgb(naip_p)
    prob   = r["prob_map"]
    mask   = r["mask"]
    axes2[row][0].imshow(rgb)
    axes2[row][0].set_title(f"{site}\nNAIP RGB", fontsize=8)
    axes2[row][0].axis("off")
    im = axes2[row][1].imshow(prob, cmap="RdYlGn", vmin=0, vmax=1)
    cbar2 = plt.colorbar(im, ax=axes2[row][1], fraction=0.046, pad=0.04)
    cbar2.ax.tick_params(labelsize=6)
    axes2[row][1].set_title(f"V12 Probability\np50={r['p50']}  p95={r['p95']}", fontsize=8)
    axes2[row][1].axis("off")
    draw_mask_panel(axes2[row][2], mask, r, meta, fontsize=8)
fig2.suptitle("V12 XGBoost Shrub Detection -- All Sites", fontsize=12, y=1.01)
fig2.tight_layout()
overview_path = OUT_ROOT / "all_sites_overview.png"
fig2.savefig(overview_path, dpi=150, bbox_inches="tight")
plt.show()
log(f"  Saved combined overview : {overview_path}")

# -- Probability histograms ------------------------------------------------
fig3, axes3 = plt.subplots(2, 3, figsize=(14, 7))
axes3 = axes3.flatten()
for i, r in enumerate(ok_results):
    ax = axes3[i]
    ax.hist(r["prob_map"].flatten(), bins=50, color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(meta["cls_threshold"], color="red", linestyle="--",
               label=f"thr={meta['cls_threshold']}")
    ax.set_title(r["site"], fontsize=9)
    ax.set_xlabel("Shrub probability")
    ax.set_ylabel("Pixel count")
    ax.legend(fontsize=8)
for j in range(len(ok_results), 6):
    axes3[j].set_visible(False)
fig3.suptitle("V12 Probability Distributions -- All Sites", fontsize=11)
fig3.tight_layout()
hist_path = OUT_ROOT / "probability_histograms.png"
fig3.savefig(hist_path, dpi=150, bbox_inches="tight")
plt.show()
log(f"  Saved histograms: {hist_path}")


In [ ]:
log_section(f"Run complete  -  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log(f"  Log file : {LOG_PATH.resolve()}")
log(f"  Outputs  : {OUT_ROOT.resolve()}")

# Flush all handlers so the file is fully written before we read it
for h in logger.handlers:
    h.flush()

# Display full log in the notebook
log_text = LOG_PATH.read_text(encoding="utf-8")
print("\n" + "=" * 70)
print(f"  FULL RUN LOG  -  {LOG_PATH.name}")
print("=" * 70)
print(log_text)

In [ ]:
# List all log files from all previous runs
print("All pipeline logs in logs/:")
print(f"  {'Filename':<45} {'Size':>8}")
print("-" * 56)
for p in sorted(LOG_DIR.glob("*_pipeline.log")):
    kb = p.stat().st_size / 1024
    marker = "  --- this run" if p == LOG_PATH else ""
    print(f"  {p.name:<45} {kb:>6.1f} KB{marker}")